## <div  style="color:black;  font-size:100%; text-align:center;padding:12.0px; background:#ffffff"> I am glad to welcome everyone to this exciting competition! </div>



<center>
<img src="https://i.postimg.cc/x8dWTCX3/1234-Untitled.png" width=1100>
</center>



# <div  style="color:white; border:lightgreen solid;  font-weight:bold; font-size:120%; text-align:center;padding:12.0px; background:black">1. OVERVIEW</div>


# Goal
The objective of this competition is to create an algorithm that is capable of solving **abstract reasoning** tasks. Critically, these are **novel** tasks: tasks that the algorithm has never seen before. Hence, **simply memorizing** a set of reasoning templates will **not suffice**.

The goal is to construct the output grid(s) corresponding to the test input grid(s), using 2 trials for each test input.



# Data overview
- A "grid" is a **rectangular** matrix (list of lists) of integers between 0 and 9 (**inclusive**). The smallest possible grid size is **1x1** and the largest is **30x30**
- The public evaluation set is different from the public training set (**which is significantly easier**)

The following **three datasets** are associated with the ARC Prize competition:
- Public training set
- Public evaluation set
- Private evaluation set





**PUBLIC:**
- The publicly available data is to be used for training and evaluation
- The **public training set** contains 1000 task files you will use to train your algorithm
- The **public evaluation set** contains 120 task files for to test the performance of your algorithm



**PRIVATE:**
- The **private evaluation set** contains 240 task files
- The ARC-AGI leaderboard is measured using private evaluation tasks which are privately held on Kaggle. These tasks are private to ensure models may not be trained on them. These tasks are not included in the public tasks, but they do use the **same structure and cognitive priors**
- Public training set consists of **simpler tasks** whereas the public evaluation set is roughly the **same level** of difficulty as the private test set



**DIFFICULTY OF SETS:**
- The public training set is **significantly easier** than the others (public evaluation and private evaluation set) since it contains many "curriculum" type tasks intended to demonstrate Core Knowledge systems. It's like a tutorial level
- The public evaluation sets and the private test sets are intended to be the **same difficulty**


# Evaluation
- The competition evaluates submissions on the **percentage of correct predictions** on the private evaluation set (**240 tasks**)
- For each task, you should predict **exactly 2 outputs**  for every test input grid contained in the task (attempt_1, attempt_2). **All cells** should match the expected answer. Otherwise you score will be 0.  Each task's test output has **one ground truth**
- Tasks can have **more than one test input** that needs a predicted output. But most tasks only have a single output
- The **final score** is the sum averaged of the highest score per task output divided by the total number of task test outputs. *Ex: If there are two task outputs, and one is 100% correct and the other is 0% correct, your score is 0.5*


# Additional notebooks


-  [**Visualizing all training and evaluating set**](https://www.kaggle.com/code/allegich/arc-agi-2025-visualization-all-1000-120-tasks/)

There is a visualization of all 1000 + 120 tasks, including training set (1000) and evaluating set (120).
This can be useful to get the full vision about task, and to see the true scale and complexity of the problem.

- [**JSON: Introduction, Tricks and Applying (for ARC)**](https://www.kaggle.com/code/allegich/json-introduction-tricks-and-applying)

Since the ARC tasks have JSON format, it is important to be able to manipulate them easily and quickly. 
This notebook covers various techniques and methods for working with the JSON-format. 

Also this notebook contains an example of creating a simplified **monochrome ARC dataset**. This can be useful for hypothesis testing and models training.


I hope these  notebooks will be useful and convenient for you.

# <div  style="color:white; border:lightgreen solid;  font-weight:bold; font-size:120%; text-align:center;padding:12.0px; background:black">2. DATA LOADING AND PREPARATION</div>


## Import libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from   matplotlib import colors
import seaborn as sns

import json

Loading JSON data:

In [ ]:
base_path='/kaggle/input/arc-prize-2025/'

def load_json(file_path):
    with open(file_path) as f:
        data = json.load(f)
    return data

Reading files:

In [ ]:
training_challenges =  load_json(base_path +'arc-agi_training_challenges.json')
training_solutions =   load_json(base_path +'arc-agi_training_solutions.json')

evaluation_challenges =load_json(base_path +'arc-agi_evaluation_challenges.json')
evaluation_solutions = load_json(base_path +'arc-agi_evaluation_solutions.json')


# <div  style="color:white; border:lightgreen solid;  font-weight:bold; font-size:120%; text-align:center;padding:12.0px; background:black">3. DATA EXPLORATION</div>


Training dataset has 1000, evaluation dataset has 120 task and test dataset has 240 JSON tasks:

In [ ]:
print(f'Number of training challenges = {len(training_challenges)}')

In [ ]:
print(f'Number of training solutions = {len(training_solutions)}')

In [ ]:
print(f'Number of evaluation challenges = {len(evaluation_challenges)}')

In [ ]:
print(f'Number of evaluation solutions = {len(evaluation_solutions)}')

In [ ]:
print(f'Number of test challenges = {len(test_challenges)}')

The names of the first five "training challenges" are shown below:

In [ ]:
for i in range(5):
    t=list(training_challenges)[i]
    task=training_challenges[t]
    print(f'Set #{i}, {t}')

In each task, there are **two** dictionary keys, **train** and **test**. We learn the pattern from the train input-output pairs, and then apply the pattern to the test input, to predict an output.

In [ ]:
task = training_challenges['007bbfb7']
print(task.keys())

Tasks have multiple train input-output pairs. Most tasks have a single test input-output pair, although some have more than one.

In [ ]:
n_train_pairs = len(task['train'])
n_test_pairs = len(task['test'])

print(f'task contains {n_train_pairs} training pairs')
print(f'task contains {n_test_pairs} test pairs')

Dive into the first train input-output pair, we can see the grids are expressed as 2d lists with integers 0-9:

In [ ]:
display(task['train'][0]['input'])
display(task['train'][0]['output'])

## Functions to plot input/output pairs of a task

In [ ]:
# 0:black, 1:blue, 2:red, 3:green, 4:yellow, # 5:gray, 6:magenta, 7:orange, 8:sky, 9:brown

cmap = colors.ListedColormap(
    ['#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00',
     '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'])
norm = colors.Normalize(vmin=0, vmax=9)

plt.figure(figsize=(3, 1), dpi=150)
plt.imshow([list(range(10))], cmap=cmap, norm=norm)
plt.xticks(list(range(10)))
plt.yticks([])
plt.tick_params(axis='x', color='r', length=0, grid_color='none')
    
plt.show()

In [ ]:
def plot_task(task, task_solutions, i, t, size=2.5, w1=0.9):
    t=list(training_challenges)[i]
    titleSize=16    
    num_train = len(task['train'])
    num_test  = len(task['test'])
    
    wn=num_train+num_test
    fig, axs  = plt.subplots(2, wn, figsize=(size*wn,2*size))
    plt.suptitle(f'Task #{i}, {t}', fontsize=titleSize, fontweight='bold', y=1, color = '#eeeeee')
   
    '''train:'''
    for j in range(num_train):     
        plot_one(axs[0, j], j,task, 'train', 'input',  w=w1)
        plot_one(axs[1, j], j,task, 'train', 'output', w=w1)
    
    '''test:'''
    for k in range(num_test):
        plot_one(axs[0, j+k+1], k, task, 'test', 'input', w=w1)
        task['test'][k]['output'] = task_solutions[k]
        plot_one(axs[1, j+k+1], k, task, 'test', 'output', w=w1)
    
    axs[1, j+1].set_xticklabels([])
    axs[1, j+1].set_yticklabels([])
    axs[1, j+1] = plt.figure(1).add_subplot(111)
    axs[1, j+1].set_xlim([0, wn])
    
    '''Separators:'''
    colorSeparator = 'white'
    for m in range(1, wn):
        axs[1, j+1].plot([m,m],[0,1],'--', linewidth=1, color = colorSeparator)
    axs[1, j+1].plot([num_train,num_train],[0,1],'-', linewidth=3, color = colorSeparator)

    axs[1, j+1].axis("off")

    '''Frame and background:'''
    fig.patch.set_linewidth(5) #widthframe
    fig.patch.set_edgecolor('black') #colorframe
    fig.patch.set_facecolor('#444444') #background
   
    plt.tight_layout()
    
    print(f'#{i}, {t}') # for fast and convinience search
    plt.show()  
   
def plot_one(ax, i, task, train_or_test, input_or_output, solution=None, w=0.8):
    fs=12
    input_matrix = task[train_or_test][i][input_or_output]
    ax.imshow(input_matrix, cmap=cmap, norm=norm)
    
    #ax.grid(True, which = 'both',color = 'lightgrey', linewidth = 1.0)
    plt.setp(plt.gcf().get_axes(), xticklabels=[], yticklabels=[])
    ax.set_xticks([x-0.5 for x in range(1 + len(input_matrix[0]))])
    ax.set_yticks([x-0.5 for x in range(1 + len(input_matrix))])
    
    '''Grid:'''
    ax.grid(visible= True, which = 'both', color = '#666666', linewidth = w)
    
    ax.tick_params(axis='both', color='none', length=0)
   
    '''sub title:'''
    ax.set_title(train_or_test + ' ' + input_or_output, fontsize=fs, color = '#dddddd')


# Visualization Training set (first 20)

In [ ]:
for i in range(0,20):
    t=list(training_challenges)[i]
    task=training_challenges[t] 
    task_solution = training_solutions[t]
    plot_task(task,  task_solution, i, t)
    

### There is a full visualization of all 400 training tasks:

- [Visualizing **training** set](https://www.kaggle.com/code/allegich/arc-2024-show-all-400-tasks-training-set/)

# Visualization Evaluating set (first 20)

In [ ]:
for i in range(0,20):
    t=list(evaluation_challenges)[i]
    task=evaluation_challenges[t]
    task_solution = evaluation_solutions[t]
    plot_task(task,  task_solution, i, t)

I hope these  notebook will be useful for you.

## <div  style="color:#444444;  font-weight:bold; font-size:100%;font-family: monospace; text-align:center;padding:12.0px; background:#ffffff"> Thank you for your attention! <br> Please upvote this notebook if you like it. <br> It motivates me to produce more interesting and quality content) </div>